<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/CNNPretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from google.colab import userdata,files

In [24]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("Kaggle API activated via Secrets!")

Kaggle API activated via Secrets!


In [26]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
140k-real-and-fake-faces.zip: Skipping, found more recently modified local copy (use --force to force download)


In [27]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/00276TOPP4.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [28]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [32]:
train_gen = ImageDataGenerator(rescale= 1./255)
test_gen = ImageDataGenerator(rescale= 1./255)
validation_gen = ImageDataGenerator(rescale= 1./255)

In [36]:
train_data = train_gen.flow_from_directory(train_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

test_data = train_gen.flow_from_directory(test_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

validation_data = train_gen.flow_from_directory(validation_dir,
                                           target_size=(224,224),
                                           batch_size= 32,
                                           class_mode='binary')

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [34]:
# 1. Load the "Advanced" Brain (No Head, ImageNet Weights)
base_cnn = tf.keras.applications.EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False, # This is the "No Head" surgery
    weights='imagenet',
    pooling='avg'      # This automatically flattens the data for us!
)

# 2. Freeze the Brain (Phase 1)
base_cnn.trainable = False

# 3. Build the Skyscraper
model = tf.keras.Sequential([
    base_cnn,
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,247,505 (23.83 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [35]:
print(base_cnn.input_shape)

(None, 224, 224, 3)


In [38]:
# 1. Use a standard Learning Rate for the new "Head"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# 2. Launch the first 5 epochs
# I recommend setting workers=4 to speed up image loading
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5,
    verbose=2
)

Epoch 1/5
3125/3125 - 308s - 98ms/step - accuracy: 0.5006 - loss: 0.6939 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/5
3125/3125 - 247s - 79ms/step - accuracy: 0.4985 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/5
3125/3125 - 209s - 67ms/step - accuracy: 0.5017 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 4/5
3125/3125 - 209s - 67ms/step - accuracy: 0.5002 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 5/5
3125/3125 - 238s - 76ms/step - accuracy: 0.4985 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
